In [23]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

In [2]:
folder = "../Preprocessing-FeatureExtraction/cleaned-data/"
csv_files = glob.glob(os.path.join(folder, "*.csv"))

dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    df['state'] = os.path.splitext(os.path.basename(file))[0] 
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
df = data.dropna()
df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

print(f"Loaded {len(df)} reviews from {len(dfs)} states.")

Loaded 5222860 reviews from 20 states.


/var/folders/cl/5mfvhcls2nv2h9gy15m04b640000gn/T/ipykernel_6814/3678554632.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))


In [21]:
df.columns

Index(['stars', 'text', 'review_length', 'num_exclamations', 'num_caps_words',
       'clean_text', 'state', 'sentiment'],
      dtype='object')

In [43]:
sample_size = 500000
data_idx = df.sample(sample_size, random_state=42).index

# Build X (as a 2D array / DataFrame) and labels
X = df.loc[data_idx, ['clean_text', 'review_length', 'num_exclamations', 'num_caps_words']]
y_star = df.loc[data_idx, 'stars'].values
y_sent = df.loc[data_idx, 'sentiment'].values

# Single call to train_test_split to keep everything aligned
X_train, X_test, y_star_train, y_star_test, y_sent_train, y_sent_test = train_test_split(
    X, y_star, y_sent,
    test_size=0.10,
    random_state=42,
    stratify=y_star 
)

# Extract text and numeric columns separately
X_train_text = X_train['clean_text']
X_train_num = X_train.drop(columns=['clean_text']).astype(float)
X_test_text = X_test['clean_text']
X_test_num = X_test.drop(columns=['clean_text']).astype(float)

# TF-IDF on text
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

X_train_combined = hstack([X_train_tfidf, csr_matrix(X_train_num.values)])
X_test_combined  = hstack([X_test_tfidf,  csr_matrix(X_test_num.values)])

In [45]:
# Layer 1 — Sentiment Classifier
print("\nTraining sentiment classifier...")
sentiment_clf = LogisticRegression(max_iter=1000, solver="saga")
sentiment_clf.fit(X_train_combined, y_sent_train)

y_pred_sent = sentiment_clf.predict(X_test_combined)
print("\n=== Sentiment Classification Report ===")
print(classification_report(y_sent_test, y_pred_sent, digits=3))
print(f"Sentiment Accuracy: {accuracy_score(y_sent_test, y_pred_sent):.4f}")

# Get soft sentiment probabilities
train_sent_probs = sentiment_clf.predict_proba(X_train_combined) 
test_sent_probs = sentiment_clf.predict_proba(X_test_combined) 
# train_sent_probs_soft = soften_probs(train_sent_probs, temperature=3.0)
# test_sent_probs_soft  = soften_probs(test_sent_probs, temperature=3.0)

# train_sent_strength = train_sent_probs[:, 2] - train_sent_probs[:, 0]
# test_sent_strength  = test_sent_probs[:, 2] - test_sent_probs[:, 0]


Training sentiment classifier...


/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



=== Sentiment Classification Report ===
              precision    recall  f1-score   support

           0      0.831     0.584     0.686     10241
           1      0.600     0.007     0.014      5605
           2      0.790     0.989     0.879     34154

    accuracy                          0.796     50000
   macro avg      0.741     0.527     0.526     50000
weighted avg      0.777     0.796     0.742     50000

Sentiment Accuracy: 0.7961


In [17]:
def soften_probs(probs, temperature=2.0):
    # Raise probs to 1/T and re-normalize
    softened = np.exp(np.log(probs + 1e-9) / temperature)
    softened /= softened.sum(axis=1, keepdims=True)
    return softened

In [18]:
# Layer 1 — Sentiment Classifier
print("\nTraining sentiment classifier...")
sentiment_clf = LogisticRegression(max_iter=1000, solver="saga")
sentiment_clf.fit(X_train_tfidf, y_sent_train)

y_pred_sent = sentiment_clf.predict(X_test_tfidf)
print("\n=== Sentiment Classification Report ===")
print(classification_report(y_sent_test, y_pred_sent, digits=3))
print(f"Sentiment Accuracy: {accuracy_score(y_sent_test, y_pred_sent):.4f}")

# Get soft sentiment probabilities
train_sent_probs = sentiment_clf.predict_proba(X_train_tfidf) 
test_sent_probs = sentiment_clf.predict_proba(X_test_tfidf) 
train_sent_probs_soft = soften_probs(train_sent_probs, temperature=3.0)
test_sent_probs_soft  = soften_probs(test_sent_probs, temperature=3.0)

train_sent_strength = train_sent_probs[:, 2] - train_sent_probs[:, 0]
test_sent_strength  = test_sent_probs[:, 2] - test_sent_probs[:, 0]


Training sentiment classifier...

=== Sentiment Classification Report ===
              precision    recall  f1-score   support

           0      0.828     0.859     0.843     10241
           1      0.576     0.346     0.433      5605
           2      0.914     0.964     0.938     34154

    accuracy                          0.873     50000
   macro avg      0.773     0.723     0.738     50000
weighted avg      0.859     0.873     0.862     50000

Sentiment Accuracy: 0.8732


In [19]:
for idx, entry in enumerate(train_sent_probs_soft[0:10]):
    print(entry)
    print(y_sent_train[idx])
    print()
    

[0.1801642  0.27935556 0.54048025]
1

[0.19471702 0.29908034 0.50620264]
2

[0.4610677  0.25914372 0.27978858]
0

[0.33762936 0.28853781 0.37383283]
0

[0.05077992 0.15408003 0.79514005]
2

[0.04710489 0.09521149 0.85768362]
2

[0.14524252 0.28185749 0.57289999]
2

[0.14172254 0.22052871 0.63774875]
2

[0.21423314 0.36422436 0.4215425 ]
2

[0.02274862 0.20531871 0.77193267]
1



In [20]:
# Layer 2 — Star Rating Classifier
print("\nTraining star rating classifier with soft sentiment routing...")

# X_train_star = hstack([X_train_tfidf, csr_matrix(train_sent_probs_soft)])
# X_test_star = hstack([X_test_tfidf, csr_matrix(test_sent_probs_soft)])

X_train_star = hstack([
    X_train_tfidf,
    csr_matrix(train_sent_probs_soft),
    csr_matrix(train_sent_strength.reshape(-1, 1))
])

X_test_star = hstack([
    X_test_tfidf,
    csr_matrix(test_sent_probs_soft),
    csr_matrix(test_sent_strength.reshape(-1, 1))
])

print("training")

star_clf = LogisticRegression(max_iter=1000, solver="saga")
star_clf.fit(X_train_star, y_star_train)

y_pred_star = star_clf.predict(X_test_star)
print("\n=== Star Rating Classification Report ===")
print(classification_report(y_star_test, y_pred_star, digits=3))
print(f"Star Rating Accuracy: {accuracy_score(y_star_test, y_pred_star):.4f}")


Training star rating classifier with soft sentiment routing...
training

=== Star Rating Classification Report ===
              precision    recall  f1-score   support

         1.0      0.000     0.000     0.000      6055
         2.0      0.000     0.000     0.000      4186
         3.0      0.000     0.000     0.000      5605
         4.0      0.288     0.008     0.015     11892
         5.0      0.445     0.994     0.615     22262

    accuracy                          0.444     50000
   macro avg      0.147     0.200     0.126     50000
weighted avg      0.267     0.444     0.277     50000

Star Rating Accuracy: 0.4444


/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
